In [ ]:
#The code below is based on  DAMASK 3.0.0. If you use the newest version, the part for config material and load dream3D file may have changed. 
#The following code needs the inputs:1. Dream3D file. 2. The number of grids in the x,y,z directions of the RVE.
#3.The files of material property.
import damask
import h5py
import sys
import os
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from numpy import array
from numpy.linalg import norm

#The number of grids in the x,y,z directions of the RVE
cell_num_1=120
cell_num_2=120
cell_num_3=120

#The material properties files
PropertyFileFerrite='material_phase_Ferrite.yaml'
PropertyFileCarbide='material_phase_Carbide.yaml'

#For the tensile tests at different directions, we normally consider 0-90 degress. For my research,the interval
#is 15 degrees.
degree_inc=15
TestsNumber=int(90/degree_inc)+1

In [ ]:
#Define load case. In the case below two load steps are defined, with the first one-a small load, and the second one-the big load
def inversion(l,fill=0):
    return [inversion(i,fill) if isinstance(i,list) else\
            fill if i == 'x' else 'x' for i in l]
load_case = damask.Config(solver={'mechanical':'spectral_basic'},
                          loadstep=[])
#Tensile tests along x direction, the strain rate is defined.
dot_F = [[0.0001, 0 , 0 ],
     [   0,'x', 0 ],
     [   0, 0 ,'x']]
# The parameter t is the time in seconds, N is the number of increments, f_out is the frequency for the output of increments. 
loadstep = {'boundary_conditions':{'mechanical':{'dot_F':dot_F,
                                                 'dot_P':inversion(dot_F)}},
                                   'discretization':{'t':40,'N':10},'f_out':1}
load_case['loadstep'].append(loadstep)
dot_F = [[0.0001, 0 , 0 ],
     [   0,'x', 0 ],
     [   0, 0 ,'x']]

loadstep = {'boundary_conditions':{'mechanical':{'dot_F':dot_F,
                                                 'dot_P':inversion(dot_F)}},
                                   'discretization':{'t':60,'N':20},'f_out':1}
load_case['loadstep'].append(loadstep)
# load_case.save('tension.yaml')
load_case.save('load.yaml')

In [ ]:
#Import grid and geometry from .dream3d file.
filename_dream3d='RealRVE_1_1.dream3d'
#Import config_material from .dream3d file.
config_material = damask.ConfigMaterial.load_DREAM3D(filename_dream3d,grain_data="Grain Data")
#Assign the material params to the corresponding phase.
config_material['phase']['Ferrite'] = damask.ConfigMaterial.load(PropertyFileFerrite)
config_material['phase']['Carbide'] = damask.ConfigMaterial.load(PropertyFileCarbide)
config_material['phase']['Unknown Phase Type'] = damask.ConfigMaterial.load(PropertyFileFerrite)
config_material['homogenization']['direct'] = {'N_constituents':1,'mechanical':{'type':'pass'}}
config_material.save()
#Import grid from .dream3d file.
g=damask.Grid.load_DREAM3D(filename_dream3d,feature_IDs="FeatureIds")
#CutRVE=[cell_num_1,cell_num_2,cell_num_3]
#g=g.canvas(CutRVE)
#g=g.scale(CutRVE)
g.save('geometry')

In [ ]:
#The following code conducts tensile test at different directions with respect to rolling direction.
#7 tensile test from 0-90 degrees are conducted below with 15 degrees' increment.
#The description of the method are in the Section 3.2 of paper(https://doi.org/10.1016/j.jmrt.2025.05.235)
all_data = pd.DataFrame()
for a in range(0,TestsNumber):
    degree=a*degree_inc
    #Sometimes, after rotation, the cell's number along x or y axis is odd. In order to find the center of enlarged RVE， we should make the cells even. So, add one if the cell number is odd.
    def add_one_if_odd(num):
        if num % 2 != 0:
            return num + 1
        else:
            return num
    
    
    #When the "Canvas" function is used, the Offset should be defined. This function "OffsetVector" is used to define the vector of Offset for "Canvas" function
    def OffsetVector(string):
        ga=[]
        for i in range(3):
            ga.append(add_one_if_odd(string.cells[i]))
        
        gt=[]
        
        gt.append((ga[0] - cell_num_1) / 2)
        gt.append((ga[1] - cell_num_2) / 2)
        gt.append((ga[2] - cell_num_3) / 2)
        return gt
    
    #Use "mirror" to enlarge the RVE 4 times in x and y directions 
    
    g=g.mirror('xy',True)
    g=g.mirror('xy',True)
    #Cut the enlarged RVE to the 3 time of original RVE
    g=g.canvas([3*cell_num_1,3*cell_num_2,cell_num_3],[0,0,0])
    g.save('geometry1')
    #Cut the enlarged RVE to the the same size as original RVE
    g_0=g.canvas([cell_num_1,cell_num_2,cell_num_3],[cell_num_1,cell_num_2,0])
    g_0.save('geometry0')
    #Rotate RVE by different angles.
    de=degree/180*np.pi
    names = locals()
    names['g' + str(de)] = g.rotate(damask.Rotation.from_Euler_angles(np.array([de,0,0])), fill=False)
    names['g' + str(de)].save('geometry2')
    #Cut the rotated RVE
    names['g' + str(de)] = names['g' + str(de)].canvas([cell_num_1,cell_num_2,cell_num_3],OffsetVector(names['g' + str(de)]))
    #Save the rotated RVE, the size is the same as original one
    names['g' + str(de)].save(f"geometry_{degree}_1") 
    names['g' + str(de)] =names['g' + str(de)].flip('x')
    names['g' + str(de)] =names['g' + str(de)].flip('y')
    #names['g' + str(de)].save(f"geometry_{degree}")
    names['g' + str(de)].save('geometry_Rotated')
    #The above code rotates the geometry of RVE, the following code rotates the orientations of each grides.
    config_material_Origin = damask.ConfigMaterial.load_DREAM3D(filename_dream3d,grain_data="Grain Data")
    Material_quantity=0
    for all in config_material_Origin['material']:
        Material_quantity+=1
    #The grain orientations exported from Dream3D are in "array[]" form but "quatenion[]" form is used in damask.rotation function, the next for loop is used to transfer "array[]" form to "quatenion[]" form.
    for i in range(Material_quantity):
        config_material_Origin['material'][i]['constituents'][0]['O']=damask.Rotation.from_quaternion(config_material_Origin['material'][i]['constituents'][0]['O'])
    #Rotate grian orientations and save them in "material.yaml"
    for i in range(Material_quantity): 
        Orientation_Origin=config_material_Origin['material'][i]['constituents'][0]['O']
        Rotation_angles=damask.Rotation.from_Euler_angles(np.array([np.pi*2-de,0,0]))
        Orientation_New=Orientation_Origin*Rotation_angles
        config_material['material'][i]['constituents'][0]['O']=Orientation_New
    config_material.save()
    #Run the simulation at this direction
    ! mpiexec -n 40 DAMASK_grid --load load.yaml --geom geometry_Rotated.vti --material material.yaml | tail -n 2
    old_name = f"geometry_Rotated_load.hdf5"
    new_name = f"geometry_{degree}_load.hdf5"    
    # Renaming the file
    os.rename(old_name, new_name)
    results = damask.Result(f"geometry_{degree}_load.hdf5")
    results.add_stress_Cauchy()
    results.add_strain()
    results.add_strain('F_p','U')
    results.add_equivalent_Mises('sigma')
    results.add_equivalent_Mises('epsilon_V^0.0(F)')
    # Calculate Von Mises stress
    sigma_0 = [np.average(s) for s in results.place('sigma_vM').values()]
    # Calculate Von Mises strain
    epsilon_0 = [np.average(e) for e in results.place('epsilon_V^0.0(F)_vM').values()]
    results.add_equivalent_Mises('epsilon_U^0.0(F_p)')
    epsilon_avg = np.array([np.average(eps,0) for eps in results.place('epsilon_U^0.0(F_p)').values()])
    # Calculate r value
    r = epsilon_avg[:,1,1]/epsilon_avg[:,2,2]
    #               ^                  ^add the increment number of yield point
    #Plastic von mises strain
    s = np.array([np.average(strain) for strain in results.place('epsilon_U^0.0(F_p)_vM').values()])
    csv_file = "r.csv"
    
    df_new = pd.DataFrame({
        f"PlasticStrain_{degree}": pd.Series(s),
        f"r_{degree}": pd.Series(r),
        f"epsilon_{degree}": pd.Series(epsilon_0),
        f"sigma_{degree}": pd.Series(sigma_0)
    })

    all_data = pd.concat([all_data, df_new], axis=1)
    results.add_IPF_color(np.array([0,0,1]))
    #Delet the colors of carbides so that carbides are shown black in Paraview
    with h5py.File(f"geometry_{degree}_load.hdf5" ,  "a") as f:
        del f['increment_0']['phase']['Carbide']['mechanical']['IPFcolor_(0 0 1)']
    #Export VTI file
    results.export_VTK()
    #os.remove(f"geometry_{degree}_load.hdf5" )
all_data.to_csv(csv_file, index=False, sep=",")
